# 04 — Test Inference and Submission

**Run after all 5 folds of `03_train.ipynb` have completed.**

What this notebook does:
1. Loads all 5 fold checkpoints from Drive
2. Computes full OOF threshold sweep (requires pair tensors from notebook 02)
3. Loads test embeddings from Drive
4. Builds test FAISS indices and runs dual-modality blocking
5. Scores each test S1 entity's candidates using the 5-model ensemble
6. Writes `matching_results.tsv` and `candidate_pairs.tsv`
7. Runs `validate_submission.py`

In [ ]:
# ── Colab setup ──────────────────────────────────────────────────────────────
!pip install -q transformers sentencepiece faiss-cpu

import os
try:
    import google.colab
    COLAB = True
except ImportError:
    COLAB = False

BASE_PATH = '/content/drive/MyDrive/Amazon/student_resource 2'  # EDIT

if COLAB and BASE_PATH.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

DATA_DIR = os.path.expanduser(BASE_PATH)
assert os.path.isdir(DATA_DIR), f'Folder not found at {DATA_DIR}'
print('Dataset OK at', DATA_DIR)

WORK_ROOT = '/content/er'
os.makedirs(WORK_ROOT, exist_ok=True)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('Colab ready.')

## Section 0 — Configuration

In [ ]:
from pathlib import Path
from collections import defaultdict
import pandas as pd
import numpy as np
import torch
import json
import csv
import gc
import faiss
import subprocess
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold
from tqdm import tqdm

In [ ]:
try:
    COLAB
except NameError:
    COLAB = False; DATA_DIR = None; WORK_ROOT = None

if COLAB:
    DATASET_ROOT = Path(DATA_DIR)
    ROOT = Path(WORK_ROOT)
else:
    ROOT = Path.cwd()
    if not (ROOT / 'dataset').exists():
        ROOT = Path('..').resolve()
    DATASET_ROOT = ROOT

TEST      = DATASET_ROOT / 'dataset/test'
EMB       = DATASET_ROOT / 'output/embeddings'
CKPT_DIR  = DATASET_ROOT / 'output/checkpoints'
OUT       = ROOT / 'output'; OUT.mkdir(exist_ok=True)
WORK      = ROOT / 'working'; WORK.mkdir(exist_ok=True)
UTILS_DIR = DATASET_ROOT / 'utils'

# ── Config ───────────────────────────────────────────────────────────────────
SEED            = 42
ENCODER_DIM     = 768
NAME_DIM        = ENCODER_DIM
ADDR_DIM        = ENCODER_DIM
TOWER_DIM       = 512
BATCH_SIZE      = 128
EPOCHS          = 150
N_FOLDS         = 5
TOP_K_FAISS     = 80
QUANTIZED_INDEX = True

is_cuda_available = torch.cuda.is_available()
DEVICE = torch.device('cuda' if is_cuda_available else 'cpu')
print(f'Device: {DEVICE}')
np.random.seed(SEED); torch.manual_seed(SEED)

## Check: verify all 5 fold checkpoints exist

In [ ]:
missing = []
for fold in range(1, N_FOLDS + 1):
    p = CKPT_DIR / f'fold_{fold}_best.pt'
    if not p.exists():
        missing.append(str(p))

if missing:
    raise FileNotFoundError(
        f'Missing fold checkpoints — run 03_train.ipynb for all folds first:\n' +
        '\n'.join(missing)
    )
print('All 5 fold checkpoints found. Proceeding...')

## Model and Dataset Definitions (must match 03_train.ipynb)

In [ ]:
class PairDataset(Dataset):
    def __init__(self, name_inputs, addr_inputs, scalars, labels):
        self.name_inputs = torch.from_numpy(name_inputs)
        self.addr_inputs = torch.from_numpy(addr_inputs)
        self.scalars     = torch.from_numpy(scalars)
        self.labels      = torch.from_numpy(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (self.name_inputs[idx], self.addr_inputs[idx],
                self.scalars[idx], self.labels[idx])


class ModalTowerAdapter(nn.Module):
    def __init__(self, name_dim=NAME_DIM*2, addr_dim=ADDR_DIM*2,
                 tower_dim=TOWER_DIM, scalar_dim=3, dropout=0.1):
        super().__init__()

        def make_tower(in_dim):
            return nn.Sequential(
                nn.Linear(in_dim, tower_dim), nn.ReLU(), nn.LayerNorm(tower_dim), nn.Dropout(dropout),
                nn.Linear(tower_dim, tower_dim), nn.ReLU(), nn.LayerNorm(tower_dim), nn.Dropout(dropout),
                nn.Linear(tower_dim, tower_dim), nn.ReLU(), nn.LayerNorm(tower_dim), nn.Dropout(dropout),
            )

        self.name_tower = make_tower(name_dim)
        self.addr_tower = make_tower(addr_dim)
        fusion_dim = tower_dim + tower_dim + scalar_dim
        self.adapter = nn.Sequential(
            nn.Linear(fusion_dim, 1024), nn.ReLU(), nn.LayerNorm(1024), nn.Dropout(dropout),
            nn.Linear(1024, 512),        nn.ReLU(), nn.LayerNorm(512),  nn.Dropout(dropout),
            nn.Linear(512, 256),         nn.ReLU(), nn.LayerNorm(256),  nn.Dropout(dropout),
            nn.Linear(256, 1)
        )

    def forward(self, name_input, addr_input, scalars):
        fused = torch.cat([self.name_tower(name_input),
                           self.addr_tower(addr_input), scalars], dim=1)
        return self.adapter(fused).squeeze(1)

## FAISS Utilities

In [ ]:
def build_faiss_index(embeddings_fp16, use_gpu=False, quantize=QUANTIZED_INDEX):
    emb = embeddings_fp16.astype(np.float32)
    D = emb.shape[1]
    if quantize and not use_gpu:
        index = faiss.IndexScalarQuantizer(D, faiss.ScalarQuantizer.QT_8bit)
    else:
        index = faiss.IndexFlatIP(D)
    if use_gpu and faiss.get_num_gpus() > 0:
        res = faiss.StandardGpuResources()
        index = faiss.index_cpu_to_gpu(res, 0, index)
    if isinstance(index, faiss.IndexScalarQuantizer):
        index.train(emb)
    index.add(emb)
    return index


def dual_faiss_search(s1_names, s1_addrs, name_index, addr_index, target_ids, k=80):
    name_scores, name_indices = name_index.search(s1_names.astype(np.float32), k)
    addr_scores, addr_indices = addr_index.search(s1_addrs.astype(np.float32), k)
    result = {}
    for i in range(len(s1_names)):
        candidates = set()
        for idx in name_indices[i]:
            if 0 <= idx < len(target_ids):
                candidates.add(target_ids[idx])
        for idx in addr_indices[i]:
            if 0 <= idx < len(target_ids):
                candidates.add(target_ids[idx])
        result[i] = candidates
    return result


def country_filter(candidates_dict, s1_countries, target_country_map, allow_mismatch=False):
    if allow_mismatch:
        return candidates_dict
    filtered = {}
    for local_idx, cand_set in candidates_dict.items():
        s1_country = str(s1_countries[local_idx]).lower().strip() if local_idx < len(s1_countries) else ''
        filtered_cands = set()
        for cid in cand_set:
            cand_country = str(target_country_map.get(cid, '')).lower().strip()
            if s1_country == cand_country or s1_country == '' or cand_country == '':
                filtered_cands.add(cid)
        filtered[local_idx] = filtered_cands
    return filtered


def load_country_map(paths):
    result = {}
    for p in paths:
        for chunk in pd.read_csv(p, sep='\t', chunksize=100_000,
                                  usecols=['entity_id', 'country'], low_memory=False):
            for _, row in chunk.iterrows():
                result[str(row['entity_id'])] = str(row['country'])
    return result


def pair_features(s1_name_emb, s1_addr_emb, cand_name_emb, cand_addr_emb,
                   s1_country, cand_country):
    name_input = np.concatenate([s1_name_emb, cand_name_emb]).astype(np.float32)
    addr_input = np.concatenate([s1_addr_emb, cand_addr_emb]).astype(np.float32)
    cos_name   = float(np.dot(s1_name_emb, cand_name_emb))
    cos_addr   = float(np.dot(s1_addr_emb, cand_addr_emb))
    country_eq = float(str(s1_country).lower().strip() == str(cand_country).lower().strip())
    scalars = np.array([cos_name, cos_addr, country_eq], dtype=np.float32)
    return name_input, addr_input, scalars

## macro F0.5

In [ ]:
def macro_f05(labels, predictions, groups, all_s1_ids=None):
    by_source = defaultdict(list)
    for i, sid in enumerate(groups):
        by_source[sid].append(i)
    if all_s1_ids is not None:
        for sid in all_s1_ids:
            if sid not in by_source:
                by_source[sid] = []
    scores = []
    for sid, indices in by_source.items():
        if not indices:
            scores.append(1.0)
            continue
        t  = labels[indices].astype(bool)
        p  = predictions[indices].astype(bool)
        tp = np.sum(t & p)
        fp = np.sum(~t & p)
        fn = np.sum(t & ~p)
        prec = tp / max(tp + fp, 1)
        rec  = tp / max(tp + fn, 1)
        if prec + rec == 0:
            scores.append(1.0 if (not t.any() and not p.any()) else 0.0)
        else:
            scores.append(1.25 * prec * rec / (0.25 * prec + rec))
    return float(np.mean(scores)) if scores else 0.0

## Load All 5 Fold Models

In [ ]:
fold_models = []
for fold in range(1, N_FOLDS + 1):
    final_path = CKPT_DIR / f'fold_{fold}_best.pt'
    saved = torch.load(final_path, map_location=DEVICE)
    model = ModalTowerAdapter().to(DEVICE)
    model.load_state_dict(saved['best_state'])
    model.eval()
    fold_models.append(model)
    print(f'Fold {fold} loaded — best val F0.5: {saved["best_score"]:.4f}')

print(f'\nAll {N_FOLDS} fold models loaded.')

## OOF Threshold Sweep

Runs over the saved pair tensors to find the optimal decision threshold.

In [ ]:
print('Loading pair tensors for OOF threshold sweep...')
all_name_inputs = np.load(EMB / 'all_name_inputs.npy')
all_addr_inputs = np.load(EMB / 'all_addr_inputs.npy')
all_scalars     = np.load(EMB / 'all_scalars.npy')
all_labels      = np.load(EMB / 'all_labels.npy')
all_groups      = np.load(EMB / 'all_groups.npy', allow_pickle=True)

kfold      = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
all_splits = list(kfold.split(all_name_inputs))
oof_probs  = np.zeros(len(all_labels), dtype=np.float32)

for fold_zero in range(N_FOLDS):
    fold = fold_zero + 1
    _, val_idx = all_splits[fold_zero]
    val_ds = PairDataset(
        all_name_inputs[val_idx], all_addr_inputs[val_idx],
        all_scalars[val_idx],     all_labels[val_idx]
    )
    loader = DataLoader(val_ds, batch_size=512, shuffle=False)
    model  = fold_models[fold_zero]
    probs  = []
    with torch.no_grad():
        for name_inp, addr_inp, scalars, _ in loader:
            logits = model(name_inp.to(DEVICE), addr_inp.to(DEVICE), scalars.to(DEVICE))
            probs.extend(torch.sigmoid(logits).cpu().numpy().tolist())
    oof_probs[val_idx] = probs
    print(f'  Fold {fold} OOF done')

best_threshold, best_oof_score = 0.5, -1.0
for t in np.linspace(0.30, 0.95, 66):
    score = macro_f05(all_labels, oof_probs >= t, all_groups)
    if score > best_oof_score:
        best_oof_score, best_threshold = score, float(t)

print(f'\nOOF macro-F0.5: {best_oof_score:.4f}  threshold: {best_threshold:.3f}')
(WORK / 'validation_metrics.json').write_text(json.dumps({
    'oof_macro_f05': best_oof_score,
    'threshold':     best_threshold,
    'folds':         N_FOLDS,
    'epochs':        EPOCHS
}, indent=2))

# Free pair tensors — no longer needed
del all_name_inputs, all_addr_inputs, all_scalars, all_labels, all_groups, oof_probs
gc.collect()

## Section 4 — Load Test Embeddings and Build Test FAISS Indices

In [ ]:
print('Loading test embeddings...')
test_s1_name = np.load(EMB / 'test_s1_name.npy')
test_s1_addr = np.load(EMB / 'test_s1_addr.npy')
test_s1_ids  = np.load(EMB / 'test_s1_name_ids.npy', allow_pickle=True).tolist()

test_s2_name = np.load(EMB / 'test_s2_name.npy')
test_s3_name = np.load(EMB / 'test_s3_name.npy')
test_s2_addr = np.load(EMB / 'test_s2_addr.npy')
test_s3_addr = np.load(EMB / 'test_s3_addr.npy')
test_s2_ids  = np.load(EMB / 'test_s2_name_ids.npy', allow_pickle=True).tolist()
test_s3_ids  = np.load(EMB / 'test_s3_name_ids.npy', allow_pickle=True).tolist()

test_target_name   = np.concatenate([test_s2_name, test_s3_name], axis=0)
test_target_addr   = np.concatenate([test_s2_addr, test_s3_addr], axis=0)
test_target_ids    = test_s2_ids + test_s3_ids
test_target_id_set = set(test_target_ids)

test_s1_country_map     = load_country_map([TEST / 'test_source1.tsv'])
test_target_country_map = load_country_map([TEST / 'test_source2.tsv', TEST / 'test_source3.tsv'])

print('Building test FAISS indices...')
test_name_index = build_faiss_index(test_target_name)
test_addr_index = build_faiss_index(test_target_addr)
print('Test setup complete.')

## Section 4 — FAISS Blocking for All Test S1 Entities

In [ ]:
BATCH = 10_000
s1_candidates = {}

for start in tqdm(range(0, len(test_s1_ids), BATCH), desc='FAISS blocking'):
    end         = min(start + BATCH, len(test_s1_ids))
    batch_name  = test_s1_name[start:end]
    batch_addr  = test_s1_addr[start:end]
    batch_countries = [test_s1_country_map.get(test_s1_ids[i], '') for i in range(start, end)]

    cands = dual_faiss_search(
        batch_name, batch_addr,
        test_name_index, test_addr_index,
        test_target_ids, k=TOP_K_FAISS
    )
    cands = country_filter(cands, batch_countries, test_target_country_map)

    for local_i, cand_set in cands.items():
        global_i = start + local_i
        s1_candidates[global_i] = [eid for eid in cand_set if eid in test_target_id_set]

print(f'FAISS blocking complete for {len(test_s1_ids):,} S1 entities.')

## Section 4 — Score Candidates and Write Submission

In [ ]:
target_id_to_idx = {eid: i for i, eid in enumerate(test_target_ids)}


def score_candidates(s1_idx, candidate_ids, fold_models, threshold):
    """
    Score all candidates for one S1 entity using ensemble of fold models.
    Returns (matched_ids, all_candidate_ids).
    """
    if not candidate_ids:
        return [], []

    s1_name_emb = test_s1_name[s1_idx].astype(np.float32)
    s1_addr_emb = test_s1_addr[s1_idx].astype(np.float32)
    s1_country  = test_s1_country_map.get(test_s1_ids[s1_idx], '')

    name_inputs, addr_inputs, scalars_list = [], [], []
    for cid in candidate_ids:
        cidx          = target_id_to_idx[cid]
        cand_name_emb = test_target_name[cidx].astype(np.float32)
        cand_addr_emb = test_target_addr[cidx].astype(np.float32)
        cand_country  = test_target_country_map.get(cid, '')
        ni, ai, sc    = pair_features(s1_name_emb, s1_addr_emb,
                                       cand_name_emb, cand_addr_emb,
                                       s1_country, cand_country)
        name_inputs.append(ni)
        addr_inputs.append(ai)
        scalars_list.append(sc)

    name_t = torch.tensor(np.array(name_inputs), dtype=torch.float32).to(DEVICE)
    addr_t = torch.tensor(np.array(addr_inputs), dtype=torch.float32).to(DEVICE)
    sc_t   = torch.tensor(np.array(scalars_list), dtype=torch.float32).to(DEVICE)

    with torch.no_grad():
        all_probs = np.mean([
            torch.sigmoid(m(name_t, addr_t, sc_t)).cpu().numpy()
            for m in fold_models
        ], axis=0)

    matched = [cid for cid, p in zip(candidate_ids, all_probs) if p >= threshold]
    return matched, candidate_ids

In [ ]:
match_path = OUT / 'matching_results.tsv'
cand_path  = OUT / 'candidate_pairs.tsv'

with open(match_path, 'w', newline='', encoding='utf-8') as mf, \
     open(cand_path,  'w', newline='', encoding='utf-8') as cf:

    mw = csv.writer(mf, delimiter='\t')
    cw = csv.writer(cf, delimiter='\t')
    mw.writerow(['source1_entity_id', 'matched_entity_ids'])
    cw.writerow(['source1_entity_id', 'candidate_entity_ids'])

    for i, s1_id in enumerate(tqdm(test_s1_ids, desc='Scoring and writing')):
        candidate_ids = s1_candidates.get(i, [])
        matched, candidates_out = score_candidates(
            i, candidate_ids, fold_models, best_threshold
        )
        matched_clean    = sorted(set(matched))
        candidates_clean = sorted(set(candidates_out))
        mw.writerow([s1_id, ','.join(matched_clean)])
        cw.writerow([s1_id, ','.join(candidates_clean)])

        if (i + 1) % 100_000 == 0:
            print(f'  Processed {i+1:,} / {len(test_s1_ids):,} S1 entities')

print(f'Wrote {match_path}')
print(f'Wrote {cand_path}')

## Section 5 — Validate Submission

In [ ]:
result = subprocess.run([
    'python3',
    str(UTILS_DIR / 'validate_submission.py'),
    '--matching',  str(match_path),
    '--candidate', str(cand_path),
    '--test-dir',  str(TEST)
], capture_output=True, text=True)

print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr)

## Done

Submission files written to `ROOT/output/`:
```
matching_results.tsv   — final predictions
candidate_pairs.tsv    — all FAISS-blocked candidates (for diagnostic)
```

Validation metrics saved to `ROOT/working/validation_metrics.json`.